# Policy composition, activation weight, and normalized payoff

Rows show population sizes, columns show agent groups, color indicates the population share assigned to the payoff-guided policy, and the x-axis is each random agent's activation weight relative to a payoff-guided agent's unit weight. The all-agent payoff is the assignment-share-weighted mean of the two policy-group payoffs. At homogeneous population endpoints, a common activation weight normalizes away, so those curves are shown as flat references.

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
from matplotlib.cm import ScalarMappable
from matplotlib.colors import BoundaryNorm, ListedColormap
from matplotlib.patches import Rectangle
from matplotlib_inline.backend_inline import set_matplotlib_formats
import numpy as np
import xarray as xr
from statsmodels.nonparametric.smoothers_lowess import lowess

import scienceplots  # noqa: F401
plt.style.use("science")
set_matplotlib_formats("svg")

extra = 3
plt.rcParams.update({
    "font.size": 14+extra,
    "axes.titlesize": 17+extra,
    "axes.labelsize": 15+extra,
    "xtick.labelsize": 12+extra,
    "ytick.labelsize": 12+extra,
    "legend.fontsize": 12+extra,
    "legend.title_fontsize": 13+extra,
    "figure.titlesize": 19+extra,
})

project_root = Path.cwd().resolve()
if not (project_root / "tasks").is_dir():
    project_root = project_root.parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from tasks.policies import load_policy_results

In [ ]:
# Replace this with the result store generated from the weighted run config.
result_store_path = project_root / "artifacts/simulation_runs/run-20260714T205859.zarr"
policy_stat = "policy_welfare.mean"

ds = load_policy_results(result_store_path, stats=[policy_stat])
policy_payoff = (
    ds[policy_stat]
    .isel(step=-1, drop=True)
    .reset_coords(drop=True)
    .copy(deep=True)
)
random_activation_weight = (
    ds["theta"]
    .sel(theta_parameter="random.agent.activation_weight", drop=True)
    .reset_coords(drop=True)
    .copy(deep=True)
)
del ds

best_response_payoff = policy_payoff.sel(policy_class="best_response", drop=True)
random_payoff = policy_payoff.sel(policy_class="random", drop=True)
best_response_proportion = policy_payoff.coords["best_response_proportion"]
everyone_payoff = (
    best_response_proportion * best_response_payoff.fillna(0.0)
    + (1.0 - best_response_proportion) * random_payoff.fillna(0.0)
)

population_groups = ("All agents", "Payoff-guided agents", "Random-action agents")
mean_normalized_payoff = (
    xr.concat(
        [everyone_payoff, best_response_payoff, random_payoff],
        dim="population_group",
    )
    .assign_coords(population_group=list(population_groups))
    .rename("mean_normalized_payoff")
)

plot_frame = (
    xr.Dataset({
        "mean_normalized_payoff": mean_normalized_payoff,
        "random_activation_weight": random_activation_weight,
    })
    .to_dataframe()
    .reset_index()
    .dropna(subset=["mean_normalized_payoff", "random_activation_weight"])
)

print(f"Loaded final-step payoffs from {result_store_path.relative_to(project_root)}")

In [ ]:
def smooth_payoff(series, *, lowess_frac=0.2, smooth_points=200):
    ordered = series.sort_values("random_activation_weight")
    x = ordered["random_activation_weight"].to_numpy(dtype=float)
    y = ordered["mean_normalized_payoff"].to_numpy(dtype=float)

    x_grid = np.linspace(x[0], x[-1], smooth_points)
    interpolation_delta = (x[-1] - x[0]) / smooth_points
    fitted = lowess(
        y,
        x,
        frac=lowess_frac,
        it=0,
        delta=interpolation_delta,
        is_sorted=True,
        return_sorted=False,
    )
    residual_variance = lowess(
        (y - fitted) ** 2,
        x,
        frac=lowess_frac,
        it=0,
        delta=interpolation_delta,
        is_sorted=True,
        return_sorted=False,
    )
    center = np.interp(x_grid, x, fitted)
    spread = np.sqrt(np.maximum(np.interp(x_grid, x, residual_variance), 0.0))
    return x_grid, center, center - spread, center + spread


def plot_payoff_grid(frame, *, lowess_frac=0.2, smooth_points=200):
    agent_counts = np.sort(frame["n_agents"].unique())
    proportions = np.sort(frame["best_response_proportion"].unique())
    weight_limits = (
        frame["random_activation_weight"].min(),
        frame["random_activation_weight"].max(),
    )
    colors = plt.colormaps["viridis"](np.linspace(0.08, 0.85, len(proportions)))
    palette = dict(zip(proportions, colors))
    fig, axes = plt.subplots(
        len(agent_counts),
        len(population_groups),
        figsize=(15, 3.5 * len(agent_counts)),
        sharex=True,
        sharey=True,
        squeeze=False,
    )

    for row, n_agents in enumerate(agent_counts):
        for column, population_group in enumerate(population_groups):
            ax = axes[row, column]
            panel = frame[
                (frame["n_agents"] == n_agents)
                & (frame["population_group"] == population_group)
            ]
            for proportion, series in panel.groupby("best_response_proportion", sort=True):
                if proportion in (0.0, 1.0):
                    payoff = series["mean_normalized_payoff"].to_numpy(dtype=float)
                    x = np.asarray(weight_limits)
                    center = np.full(2, np.mean(payoff))
                    spread = np.std(payoff)
                    lower = center - spread
                    upper = center + spread
                else:
                    x, center, lower, upper = smooth_payoff(
                        series,
                        lowess_frac=lowess_frac,
                        smooth_points=smooth_points,
                    )
                color = palette[proportion]
                ax.plot(x, center, color=color, linewidth=2)
                ax.fill_between(x, lower, upper, color=color, alpha=0.05, linewidth=0)

            ax.set_xlim(*weight_limits)
            ax.set_yscale("asinh", linear_width=0.5)
            ax.set_yticks(
                [-4, -2, -1, 0, 0.5, 1],
                labels=["-4", "-2", "-1", "0", "0.5", "1"],
            )
            ax.set_ylim(top=1)
            ax.axhline(0, color="black", linewidth=0.8, alpha=0.35)
            ax.grid(alpha=0.25)

    fig.subplots_adjust(left=0.15, right=0.87, bottom=0.08, top=0.89, hspace=0.10, wspace=0.06)
    strip_left = 0.09
    strip_width = 0.035
    for row, n_agents in enumerate(agent_counts):
        position = axes[row, 0].get_position()
        fig.add_artist(Rectangle(
            (strip_left, position.y0),
            strip_width,
            position.height,
            transform=fig.transFigure,
            facecolor="0.96",
            edgecolor="0.75",
            linewidth=0.8,
        ))
        fig.text(
            strip_left + strip_width / 2,
            position.y0 + position.height / 2,
            f"{n_agents} agents",
            ha="center",
            va="center",
            rotation=90,
            fontsize=15,
            fontweight="bold",
        )
    column_strip_bottom = axes[0, 0].get_position().y1 + 0.012
    column_strip_height = 0.035
    for column, population_group in enumerate(population_groups):
        position = axes[0, column].get_position()
        fig.add_artist(Rectangle(
            (position.x0, column_strip_bottom),
            position.width,
            column_strip_height,
            transform=fig.transFigure,
            facecolor="0.96",
            edgecolor="0.75",
            linewidth=0.8,
        ))
        fig.text(
            position.x0 + position.width / 2,
            column_strip_bottom + column_strip_height / 2,
            population_group,
            ha="center",
            va="center",
            fontsize=15,
            fontweight="bold",
        )
    colorbar_ax = fig.add_axes([0.91, 0.31, 0.018, 0.46])
    discrete_cmap = ListedColormap(colors)
    discrete_norm = BoundaryNorm(
        np.arange(len(proportions) + 1) - 0.5,
        discrete_cmap.N,
    )
    colorbar = fig.colorbar(
        ScalarMappable(norm=discrete_norm, cmap=discrete_cmap),
        cax=colorbar_ax,
        ticks=np.arange(len(proportions)),
        drawedges=True,
    )
    colorbar.ax.set_yticklabels([
        rf"{100 * value:.0f}\%" for value in proportions
    ])
    colorbar.ax.set_title(
        "Share of\npayoff-guided\nagents",
        fontsize=15,
        pad=10,
    )
    colorbar.ax.tick_params(labelsize=11, length=0)
    colorbar.outline.set_linewidth(0.8)
    colorbar.dividers.set_color("0.8")
    colorbar.dividers.set_linewidth(0.6)
    #fig.suptitle("Final-step mean normalized payoff by agent group", y=0.985)
    fig.supxlabel("Random/BR activation-weight ratio", y=0.02)
    fig.supylabel("Final-step mean normalized payoff (asinh scale)", x=0.06)
    fig.text(
        0.99,
        0.02,
        r"Shading: $\pm 1$ residual SD; endpoints are flat references",
        ha="right",
        va="bottom",
        fontsize=11,
        fontstyle="italic",
    )
    return fig, axes

In [ ]:
fig, axes = plot_payoff_grid(plot_frame)
plt.show()

In [ ]:
run_id = result_store_path.stem
figure_output_dir = project_root / "artifacts" / "figures" / run_id
figure_output_dir.mkdir(parents=True, exist_ok=True)
figure_path = figure_output_dir / "best_response_proportion_effect.pdf"
fig.savefig(figure_path, format="pdf", bbox_inches="tight")
print(f"Saved: {figure_path.relative_to(project_root)}")